This cell obtains the data from the Excel spreadsheet and prints it.

I have checked:
* whether there is an exact 1-to-1 correspondence between numbers and meanings
  * Basically true, though some of the terms are written slightly differently. This has been standardized.
* what happens when numbers are used instead of meanings
  * As long as you remember to take out the "Number" column and the "Meaning" column, you get the same size/shape result.
* whether there are any repeated rows or what looks like missing data
  * It seems like some of the rows are repeated and one of the MRCA terms doesn't have any correlated terms in attested languages.
* whether there are terms where the spacing is different
  * There are some terms where the spacing is different.
* whether the terms with different spacing are repeated
  * One of the terms with different spacing has two different spacing variants, so there are 304 terms without spaces and 305 terms with spaces.

In [1]:
import pandas as pd
import numpy as np

# clean Excel so all related terms are the same
# numbers are no longer needed because all words are the same
wordsheet = pd.read_excel('../data/Data-1.xlsx', header=1)
wordsheet.replace({'breast (n.) ': 'breast (n.)', 'rope ': 'rope (n.)', 'nasal mucus ': 'nasal mucus (n.)'}, inplace=True)
wordsheet.drop(columns=["Number"],inplace=True)

# get correspondences between meanings and MRCA
# as well as languages and words

duplicated_rows = []

word_correspondence_dict = {}
for number in range(len(wordsheet)):
    row = wordsheet.iloc[number]
    this_tuple = (row["Meaning"].strip(), row["MRCA Root"].strip())
    if this_tuple not in word_correspondence_dict:
        word_correspondence_dict[this_tuple] = {}
    else:
        duplicated_rows.append(this_tuple)
    for col in wordsheet.columns:
        if (col not in ("Meaning", "MRCA Root")
            and row[col] not in word_correspondence_dict[this_tuple]
            and row[col] is not np.nan
            and len(row[col].strip()) > 0):
            word_correspondence_dict[this_tuple][col.strip()] = row[col].strip()

'''
create DataFrame with:
* meaning of word in MRCA
* word in MRCA
* name of actual language
* actual word in that language
'''
meaning_column = []
mrca_column = []
lang_column = []
word_column = []
for meaning_mrca, langs_dict in word_correspondence_dict.items():
    for lang_word in langs_dict.items():
        meaning_column.append(meaning_mrca[0])
        mrca_column.append(meaning_mrca[1])
        lang_column.append(lang_word[0])
        word_column.append(lang_word[1])
words_df = pd.DataFrame({"Meaning": meaning_column, "MRCA": mrca_column, "Language": lang_column, "Word": word_column})

# Print some cool stuff
print("Original Size:", wordsheet.shape, wordsheet.shape[0] * wordsheet.shape[1])
print("New Size:", words_df.shape, words_df.shape[0] * words_df.shape[1])
print("Number of Cells in New Compared to Old:", (words_df.shape[0] * words_df.shape[1]) / (wordsheet.shape[0] * wordsheet.shape[1]))

print("\nExactly Duplicated Rows:")
for tup in duplicated_rows:
    print(tup)

original_set = set(wordsheet["MRCA Root"])
new_set = set(words_df["MRCA"])

print("\nDifference in Set Size")
print(len(original_set) - len(new_set))

print("\nMRCA Roots Which Strictly Weren't Used:")
completely_unused = original_set - new_set

spatially_duplicated = {(root, root.strip()) for root in completely_unused}

mrca_yes_yes, mrca_yes_no, mrca_no_new, mrca_yes_no_spacy = set(), set(), set(), set()

# checks for terms which differ by spaces
for spacy, spaceless in spatially_duplicated:
    if spaceless not in new_set:
        mrca_no_new.add(spaceless)
    elif spacy in original_set and spaceless not in original_set:
        mrca_yes_no.add(spaceless)
        mrca_yes_no_spacy.add(spacy)
    elif spacy in original_set and spaceless in original_set:
        mrca_yes_yes.add(spaceless)
    else:
        print("Weird?")

print("Spaceless:", len(mrca_yes_no), len(mrca_yes_yes), len(mrca_no_new))
print("Spacy:", len(mrca_yes_no_spacy))

print("\nMRCA Roots Which Were Written Many Times:")
print(len(new_set - original_set))


Original Size: (3167, 100) 316700
New Size: (25803, 4) 103212
Number of Cells in New Compared to Old: 0.3258983264919482

Exactly Duplicated Rows:
('fish (n.)', '*yu-er bor')
('rise (v.)', '*deg-')

Difference in Set Size
36

MRCA Roots Which Strictly Weren't Used:
Spaceless: 304 34 1
Spacy: 305

MRCA Roots Which Were Written Many Times:
304


Are "meaning" and "mrca" both necessary in words_df?

Yes, because there are homophones in MRCA. The majority of MRCA roots correspond to one term, and some of the ones which correspond to multiple terms could be a single word with a broad semantic space (a word for "nose" also meaning "nasal mucus"), but others are very obviously not the same (a word for "1PL pronoun" also meaning "warm").

In [2]:
mrca_meanings = {}
for number in range(len(words_df)):
    row = words_df.iloc[number]
    if row['MRCA'] not in mrca_meanings:
        mrca_meanings[row['MRCA']] = set()
    mrca_meanings[row['MRCA']].add(row['Meaning'])

one_meaning = 0
many_meanings = 0

for thing in mrca_meanings.items():
    if len(thing[1]) > 1:
        many_meanings += 1
    else:
        one_meaning += 1

print("One Meaning Corresponding to One MRCA Term:", one_meaning)
print("Many Meanings Corresponding to One MRCA Term:", many_meanings)

One Meaning Corresponding to One MRCA Term: 2619
Many Meanings Corresponding to One MRCA Term: 240


I created this thing by doing my own research, which is definitely more difficult to defend than their work. I basically went on Wikipedia and looked up each language to figure out the language family.

Some notable oddities were originally put in the "Unknown" section, in fear that they may skew the correlations otherwise.

I know that there is a 1-to-1 correspondence between the languages listed here and the languages in the Excel spreadsheet, because I've checked that none of the languages in the spreadsheet aren't here and none of the languages here aren't in the spreadsheet.

Assumptions Before Checking Correlation:

* Baoan
  * Bonan / Bao'an language of Bao'an people
* Dongxian
  * Mongolic Dongxiang / Santa language
* Kamnigan
  * Khamnigan language, especially considering the presence of the "Evenki (Kamnigan)" language
 
Correlation:

* Baoan
  * [('Huzhu', 220), ('Shira-Yughur', 203), ('Dagur', 198), ('Kangjia', 197), ('Buriat', 196), ('Minhe', 196), ('Oirat', 196), ('Kalmyck', 194), ('Khalkha', 193), ('Middle Mongolian (Muqaddimat al-adab)', 169)]
* Dongxian
  * [('Huzhu', 218), ('Minhe', 204), ('Shira-Yughur', 198), ('Oirat', 195), ('Dagur', 194), ('Kangjia', 194), ('Buriat', 193), ('Khalkha', 191), ('Kalmyck', 190), ('Middle Mongolian (Muqaddimat al-adab)', 172)]
* Kamnigan
  * [('Buriat', 276), ('Oirat', 268), ('Khalkha', 261), ('Kalmyck', 243), ('Dagur', 230), ('Shira-Yughur', 222), ('Huzhu', 217), ('Middle Mongolian (Muqaddimat al-adab)', 187), ('Minhe', 182), ('Kangjia', 181)]

After this, I was sure they were Mongolic.

In [3]:
# dictionary of families
# THIS MAY HAVE MISTAKES
family_dict = {'Japonic': {'Amami Asama', 'Amami Yamatohama', 'Amami Yoron', 'Fukuoka', 'Hachijo', 'Japanese', 'Kagoshima',
                           'Koshiki islands', 'Kumamoto', 'Miyako Irabu', 'Okinawa Shuri', 'Okinawa Yonamine', 'Old Japanese',
                           'Yaeyama Hatoma', 'Yaeyama Ishigaki', 'Yonaguni'},
               'Koreanic': {'(Late) Middle Korean', 'Gangwon', 'Gyeonggi', 'Hwanghae', 'Jeju', 'Northern Chungcheong', 'Northern Gyeongsang',
                            'Northern Hamgyong', 'Northern Jeolla', 'Northern Pyongan', 'Southern Chungcheong', 'Southern Gyeongsang',
                            'Southern Hamgyong', 'Southern Jeolla', 'Southern Pyongan',},
               'Mongolic': {'Baoan', 'Buriat', 'Dagur', 'Dongxian', 'Huzhu', 'Kalmyck', 'Kamnigan', 'Kangjia', 'Khalkha',
                            'Middle Mongolian (Muqaddimat al-adab)', 'Middle Mongolian (Secret History)', 'Minhe', 'Moghol', 'Oirat',
                            'Shira-Yughur'},
               'Tungusic': {'Even', 'Evenki (Kamnigan)', 'Hezhe', 'Jurchen', 'Kur-Urmi', 'Manchu', 'Nanai (Bikin)', 'Nanai (Middle Amur)',
                            'Negidal', 'Northern Evenki (Tura)', 'Northern Evenki (Tutonchany)', 'Oroch', 'Orok', 'Oroqen', 'Solon',
                            'Southern Evenki (Chiringda)', 'Stony Evenki (PT = Podkamennaya Tunguska)', 'Udihe', 'Ulcha', 'Xibe'},
               'Turkic': {'Azeri', 'BarabaTatar', 'Bashkir', 'CodexCumanicus', 'Chuvash', 'CrimeanTatar', 'Dolgan', 'Gagauz', 'KaraKalpak',
                          'KarachayBalkar', 'Karaim', 'Kazakh', 'KazanTatar', 'Khakas', 'Khalaj', 'Kirghiz', 'Kumyk', 'MiddleChulym',
                          'Nogai', 'NorthAltai', 'OldTurkic', 'Salar', 'Shor', 'SouthAltai', 'Tofa', 'Turkish', 'Turkmen', 'Tuvan', 'Uyghur',
                          'Uzbek', 'WestYugur', 'Yakut'},
              }

language_family_dict = {}

for fam, lang_list in family_dict.items():
    for lang in lang_list:
        language_family_dict[lang] = fam

family_set = set(language_family_dict)

print("Languages here but not in original:\n", len(set(words_df['Language'].unique()) - family_set))
print("Languages in original but not here:\n", len(family_set - set(words_df['Language'].unique())))

Languages here but not in original:
 0
Languages in original but not here:
 0


What are the correlations between the languages?

The goal is to have a dictionary where the keys are languages, and the values are dictionaries.
* In each of those dictionaries, the keys are other languages, and the values are integers representing how many terms descended from the MRCA they have in common.

In [4]:
terms_in_language = {}
for number in range(len(words_df)):
    row = words_df.iloc[number]
    this_lang = row['Language']
    if this_lang not in terms_in_language:
        terms_in_language[this_lang] = set()
    terms_in_language[this_lang].add((row['Meaning'], row['MRCA']))

alphabetized_languages = sorted(terms_in_language)

language_correlations = {}

for num1 in range(len(alphabetized_languages)):
    lang1 = alphabetized_languages[num1]
    fam1 = language_family_dict[lang1]
    language_correlations[lang1] = {}
    terms1 = terms_in_language[lang1]
    for num2 in range(len(alphabetized_languages)):
        lang2 = alphabetized_languages[num2]
        fam2 = language_family_dict[lang2]
        if fam1 != fam2:
            terms2 = terms_in_language[lang2]
            shared_terms = terms1 & terms2
            language_correlations[lang1][lang2] = len(shared_terms)


iterate_this = family_dict['Mongolic']

for name in iterate_this:
    print(name)
    langcor = language_correlations[name].items()
    
    thismany = 20
    cutoff = 30
    
    top_thismany = sorted(langcor, key= lambda l: -l[1])[:thismany]
    for lang, num in top_thismany:
        if num > cutoff:
            print("*", lang, "--", num, f'(family = {language_family_dict[lang]})')
        else:
            break

Khalkha
* Manchu -- 36 (family = Tungusic)
* Xibe -- 35 (family = Tungusic)
Shira-Yughur
* Xibe -- 34 (family = Tungusic)
* Manchu -- 33 (family = Tungusic)
Oirat
* Xibe -- 34 (family = Tungusic)
* Manchu -- 33 (family = Tungusic)
* Solon -- 33 (family = Tungusic)
* Tuvan -- 32 (family = Turkic)
* Bashkir -- 31 (family = Turkic)
* Kirghiz -- 31 (family = Turkic)
Kalmyck
* Manchu -- 34 (family = Tungusic)
* Xibe -- 33 (family = Tungusic)
Kangjia
Moghol
Dagur
* Manchu -- 42 (family = Tungusic)
* Xibe -- 41 (family = Tungusic)
* Oroqen -- 38 (family = Tungusic)
* Solon -- 37 (family = Tungusic)
* Hezhe -- 31 (family = Tungusic)
* Nanai (Bikin) -- 31 (family = Tungusic)
* Udihe -- 31 (family = Tungusic)
Buriat
* Manchu -- 35 (family = Tungusic)
* Xibe -- 34 (family = Tungusic)
Huzhu
* Manchu -- 34 (family = Tungusic)
* Xibe -- 31 (family = Tungusic)
Kamnigan
* Xibe -- 34 (family = Tungusic)
* Manchu -- 33 (family = Tungusic)
* Solon -- 31 (family = Tungusic)
Middle Mongolian (Secret Histor

This tries to see which MRCA terms have the most descendants in different language families.

In [5]:
term_appearances_per_family = {}
# order is: Japonic, Koreanic, Mongolic, Tungusic, Turkic
for number in range(len(words_df)):
    row = words_df.iloc[number]
    this_tuple = (row['Meaning'], row['MRCA'])
    if this_tuple not in term_appearances_per_family:
        term_appearances_per_family[this_tuple] = [0, 0, 0, 0, 0]
    this_fam = language_family_dict[row['Language']]
    match this_fam:
        case 'Japonic':
            term_appearances_per_family[this_tuple][0] += 1
        case 'Koreanic':
            term_appearances_per_family[this_tuple][1] += 1            
        case 'Mongolic':
            term_appearances_per_family[this_tuple][2] += 1
        case 'Tungusic':
            term_appearances_per_family[this_tuple][3] += 1
        case 'Turkic':
            term_appearances_per_family[this_tuple][4] += 1

print('-' * 30)

for term in term_appearances_per_family:
    appears = term_appearances_per_family[term]
    a0 = appears[0] / len(family_dict['Japonic'])
    a1 = appears[1] / len(family_dict['Koreanic'])
    a2 = appears[2] / len(family_dict['Mongolic'])
    a3 = appears[3] / len(family_dict['Tungusic'])
    a4 = appears[4] / len(family_dict['Turkic'])
    
    term_appearances_per_family[term] = [a0, a1, a2, a3, a4]

most_appearing_terms = sorted(term_appearances_per_family.items(), key=lambda l: -sum(l[1]))

print([x for x in most_appearing_terms[:20]])

------------------------------
[(('four', '*diu-'), [1.0, 0.0, 1.0, 1.0, 0.96875]), (('where?', '*xa-'), [0.0, 0.0, 0.9333333333333333, 0.95, 0.90625]), (('female (of an animal) (n.)', '*ama, eme'), [1.0, 1.0, 0.6666666666666666, 0.05, 0.03125]), (('hard', '*kata-'), [0.5625, 0.13333333333333333, 0.8666666666666667, 0.2, 0.90625]), (('bark (n.)', '*kap'), [0.9375, 1.0, 0.0, 0.0, 0.71875]), (('white', '*siara-'), [1.0, 1.0, 0.0, 0.55, 0.03125]), (('warm', '*dula-'), [0.0, 1.0, 0.6, 0.1, 0.71875]), (('1PL pronoun', '*bi-PL'), [0.0, 0.0, 0.7333333333333333, 0.65, 1.0]), (('house (n.)', '*diːba'), [0.4375, 1.0, 0.0, 0.85, 0.0]), (('1SG', '*bi'), [0.0, 0.0, 1.0, 1.0, 0.1875]), (('1PL pronoun', '*bu-PL'), [0.0, 1.0, 0.2, 0.95, 0.0]), (('that', '*ta-, te-'), [0.0, 0.0, 1.0, 0.95, 0.15625]), (('not', '*an-'), [1.0, 1.0, 0.0, 0.1, 0.0]), (('which?', '*e-'), [1.0, 1.0, 0.0, 0.05, 0.0]), (('mouth (n.)', '*ama'), [0.0, 0.0, 1.0, 1.0, 0.0]), (('blood (n.)', '*ti'), [1.0, 0.0, 1.0, 0.0, 0.0]), (('bo

This cell can be used to check things based on different words.

In [22]:
suspicious_terms = {'*ʐatʃu'}


# this goes over the different languages
for language in language_family_dict:
    if language_family_dict[language] in ['Japonic', 'Koreanic','Mongolic', 'Tungusic', 'Turkic']:
        print(f"{language}, a {language_family_dict[language]} language")

        for number in range(len(words_df)):
            row = words_df.iloc[number]
            if row["Language"] == language and row['MRCA'] in suspicious_terms:
                print(' --> '.join([row['MRCA'], row['Word']]))
        print("\n")

print("DONE!")

Koshiki islands, a Japonic language


Yaeyama Ishigaki, a Japonic language


Amami Asama, a Japonic language


Amami Yamatohama, a Japonic language


Hachijo, a Japonic language


Kumamoto, a Japonic language


Okinawa Shuri, a Japonic language


Yonaguni, a Japonic language


Okinawa Yonamine, a Japonic language


Miyako Irabu, a Japonic language


Japanese, a Japonic language


Kagoshima, a Japonic language


Old Japanese, a Japonic language


Amami Yoron, a Japonic language


Fukuoka, a Japonic language


Yaeyama Hatoma, a Japonic language


Gangwon, a Koreanic language


Southern Chungcheong, a Koreanic language


Southern Gyeongsang, a Koreanic language


(Late) Middle Korean, a Koreanic language


Hwanghae, a Koreanic language


Gyeonggi, a Koreanic language


Northern Jeolla, a Koreanic language


Northern Pyongan, a Koreanic language


Southern Pyongan, a Koreanic language


Southern Hamgyong, a Koreanic language


Jeju, a Koreanic language


Southern Jeolla, a Koreanic languag

This cell can be used to find all of the phonemes in the proposed proto-language, though it currently finds all the characters.

In [21]:
letters = {}

non_letters = {'*', '-', ' ', ':', '(', ')', ',', '~', '=', "'", '.', '>', '/', '?', '́'}

word_finder = {'ʐ', 'ʃ'}

if len(word_finder) > 0:
    print("Finding words with symbols:", ', '.join([f"'{x}'" for x in word_finder]))

for row in set(words_df['MRCA']):
    not_printed = True
    for l in row:
        if not_printed and l in word_finder:
            print(row)
            not_printed = False
        if l not in non_letters:
            if l not in letters:
                letters[l] = 0
            letters[l] += 1

print([(l[0], l[1]) for l in sorted(letters.items(), key = lambda x: -x[1])])

Finding words with symbols: 'ʃ', 'ʐ'
*ʐatʃu
[('a', 1711), ('i', 1007), ('k', 879), ('e', 877), ('u', 792), ('r', 763), ('n', 743), ('t', 668), ('o', 662), ('l', 522), ('s', 484), ('b', 483), ('g', 461), ('m', 433), ('p', 429), ('y', 302), ('d', 260), ('ü', 204), ('č', 192), ('ï', 164), ('ö', 111), ('x', 100), ('c', 96), ('ŋ', 93), ('j', 79), ('ʌ', 74), ('ə', 68), ('ɨ', 65), ('ḳ', 64), ('h', 61), ('ǝ', 51), ('š', 45), ('ǰ', 41), ('w', 37), ('C', 34), ('ɣ', 28), ('ŕ', 26), ('ĺ', 24), ('z', 23), ('ː', 22), ('χ', 16), ('ạ', 14), ('ẹ', 11), ('v', 9), ('ä', 9), ('f', 8), ('A', 7), ('V', 7), ('ɔ', 7), ('I', 5), ('ž', 5), ('̣', 5), ('ń', 4), ('T', 4), ('M', 4), ('E', 4), ('ʊ', 4), ('P', 3), ('L', 3), ('ɘ', 2), ('ĕ', 2), ('ǯ', 2), ('ī', 2), ('ź', 1), ('ē', 1), ('â', 1), ('å', 1), ('ɕ', 1), ('ʥ', 1), ('ś', 1), ('а', 1), ('ḵ', 1), ('ʂ', 1), ('ʐ', 1), ('ʃ', 1), ('G', 1)]
